# 10_report — 표·그림·결과 정리 (EV-10)

[최종 실험 계획서](../../../docs/ml/ROUTER_FINAL_EXPERIMENT_PLAN.md) 4.5절. 08(test), 있으면 09(seeds)·08a(latency)의 출력을 읽어 `report/`에 표(`results.md`, `tables/*.csv`)와 그림(`figures/*.png`)을 만든다. 결과를 다시 계산하지 않고 출력 파일의 숫자를 옮기며, 입력 SHA를 `manifest.json`에 남긴다. 09·08a가 끝나면 다시 실행하면 그 절이 채워진다.

**런타임: CPU.** 1분 안팎. 저장소의 `experiments/results/ev10-router-final-<날짜>/report/`에도 같은 것을 넣는다(로컬에서 `report.py`를 같은 인자로 실행).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
SOURCE_ZIP = RUN / 'code' / 'router-final-source-report.zip'
expected = SOURCE_ZIP.with_suffix('.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected, '소스 ZIP이 바뀌었다'
REPO = Path('/content/router-final-source')
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy==2.1.2', 'pandas==2.3.3', 'pyarrow==21.0.0'], check=True)

In [ ]:
OUT = Path('/content/report')
shutil.rmtree(OUT, ignore_errors=True)
proc = subprocess.run([sys.executable, str(REPO / 'ml/notebooks/router_final/report.py'), '--run-dir', str(RUN), '--output-dir', str(OUT)],
                      capture_output=True, text=True)
print(proc.stdout[-3000:], proc.stderr[-3000:])
assert proc.returncode == 0
FINAL = RUN / 'report'
shutil.rmtree(FINAL, ignore_errors=True)   # the report is regenerated from the 08/09/08a outputs each time
shutil.copytree(OUT, FINAL)
print(sorted(str(p.relative_to(FINAL)) for p in FINAL.rglob('*') if p.is_file()))

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown((Path('/content/report') / 'results.md').read_text().split('## 2.')[0]))
for f in ('mean-gain', 'gain-by-kappa', 'strata', 'tau-curves', 'task-types'):
    display(Image(str(Path('/content/report/figures') / f'{f}.png')))